# Game Data Manager
Manage the `games` MongoDB collection for soccer match data.

**Cells:**
1. Setup & imports
2. Init collection (indexes)
3. Drop collection
4. Push all data from Game_dataset
5. Inspect / stats

In [6]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient, ASCENDING
from pymongo.errors import BulkWriteError
from dns import resolver
from tqdm import tqdm

# Load .env from project root (3 levels up: init_game_db -> database -> scripts -> project root)
_nb_dir = Path(os.getcwd())
_project_root = _nb_dir.parent.parent.parent
load_dotenv(_project_root / '.env', override=True)

MONGO_SRV        = os.getenv('MONGO_SRV')
SOCCER_DB_NAME   = os.getenv('SOCCER_DB_NAME', 'SoccerWikiDemo')
COLLECTION_NAME  = os.getenv('GAME_COLLECTION_NAME', 'games')
DATASET_DIR      = _project_root / 'app' / 'database' / 'Game_dataset'
CSV_PATH         = _project_root / 'app' / 'database' / 'game_database.csv'

assert MONGO_SRV, 'MONGO_SRV not set — check .env'

# Use public DNS to avoid corporate resolver issues
resolver.default_resolver = resolver.Resolver(configure=False)
resolver.default_resolver.nameservers = ['8.8.8.8', '1.1.1.1']

client     = MongoClient(MONGO_SRV)
db         = client[SOCCER_DB_NAME]
collection = db[COLLECTION_NAME]

print(f'✅ Connected to MongoDB: {SOCCER_DB_NAME}.{COLLECTION_NAME}')
print(f'✅ Dataset dir: {DATASET_DIR}')

✅ Connected to MongoDB: soccerwiki.games
✅ Dataset dir: /Users/kinh.nvnamitech.io/Documents/FinalProject/app/database/Game_dataset


In [3]:
# ── Cell 2: Init collection (create indexes) ──────────────────────────────────
existing = db.list_collection_names()
if COLLECTION_NAME in existing:
    print(f'⚠️  Collection "{COLLECTION_NAME}" already exists. Creating indexes if missing...')
else:
    print(f'Creating collection "{COLLECTION_NAME}"...')

# Unique index on game_id
collection.create_index('game_id', unique=True, name='idx_game_id')

# Compound index for common search filters
collection.create_index(
    [('league', ASCENDING), ('season', ASCENDING), ('date', ASCENDING)],
    name='idx_league_season_date'
)
collection.create_index(
    [('home_team', ASCENDING), ('away_team', ASCENDING)],
    name='idx_teams'
)
collection.create_index('date', name='idx_date')

print(f'✅ Indexes created on "{COLLECTION_NAME}"')
for idx in collection.list_indexes():
    print(f'   {idx["name"]}: {idx["key"]}')

Creating collection "games"...
✅ Indexes created on "games"
   _id_: SON([('_id', 1)])
   idx_game_id: SON([('game_id', 1)])
   idx_league_season_date: SON([('league', 1), ('season', 1), ('date', 1)])
   idx_teams: SON([('home_team', 1), ('away_team', 1)])
   idx_date: SON([('date', 1)])


In [2]:
# ── Cell 3: Drop collection ───────────────────────────────────────────────────
confirm = input(f'Type DROP to confirm dropping "{COLLECTION_NAME}": ')
if confirm.strip() == 'DROP':
    db.drop_collection(COLLECTION_NAME)
    print(f'🗑️  Collection "{COLLECTION_NAME}" dropped.')
else:
    print('Aborted.')

🗑️  Collection "games" dropped.


In [7]:
# ── Cell 4: Push all data from Game_dataset ───────────────────────────────────
import re
import pandas as pd

BATCH_SIZE = 100

# Read CSV as lookup for league/season/date/teams per file_path
df_csv = pd.read_csv(CSV_PATH)
csv_index = {row['file_path']: row.to_dict() for _, row in df_csv.iterrows()}
print(f'📄 CSV loaded: {len(csv_index)} entries')


def slugify_team(name: str) -> str:
    """Normalize team name: lowercase, spaces/special chars → dashes."""
    name = name.lower().strip()
    name = re.sub(r'[^\w\s-]', '', name)   # remove punctuation except dash
    name = re.sub(r'[\s_]+', '-', name)    # spaces/underscores → dash
    name = re.sub(r'-+', '-', name)        # collapse consecutive dashes
    return name.strip('-')


def build_game_id(csv_meta: dict) -> str:
    """Build structured game_id: {league}/{season}/{date}/{home}-vs-{away}
    e.g. england_epl/2014-2015/2015-02-21/chelsea-vs-burnley
    """
    league = csv_meta.get('league', 'unknown')
    season = csv_meta.get('season', 'unknown')
    date   = csv_meta.get('date', 'unknown')
    home   = slugify_team(csv_meta.get('home_team', 'home'))
    away   = slugify_team(csv_meta.get('away_team', 'away'))
    return f"{league}/{season}/{date}/{home}-vs-{away}"


def build_document(json_path: Path, csv_meta: dict) -> dict:
    """Build a MongoDB document from a JSON file + CSV metadata."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    doc = {
        'game_id'  : build_game_id(csv_meta),
        'league'   : csv_meta.get('league', ''),
        'season'   : csv_meta.get('season', ''),
        'date'     : csv_meta.get('date', ''),
        'home_team': csv_meta.get('home_team', ''),
        'away_team': csv_meta.get('away_team', ''),
        'score'    : csv_meta.get('score', ''),
        'venue'    : csv_meta.get('venue', ''),
        'referee'  : csv_meta.get('referee', ''),
        'raw'      : data,
    }
    return doc


# Walk Game_dataset and collect all JSON paths
all_json_paths = sorted(DATASET_DIR.rglob('*.json'))
print(f'📂 Found {len(all_json_paths)} JSON files')

inserted = 0
skipped  = 0
errors   = 0
batch    = []

for json_path in tqdm(all_json_paths, desc='Building documents'):
    try:
        rel = 'database/' + str(json_path.relative_to(DATASET_DIR.parent)).replace('\\', '/')
        csv_meta = csv_index.get(rel, {})
        doc = build_document(json_path, csv_meta)
        batch.append(doc)

        if len(batch) >= BATCH_SIZE:
            try:
                result = collection.insert_many(batch, ordered=False)
                inserted += len(result.inserted_ids)
            except BulkWriteError as e:
                inserted += e.details.get('nInserted', 0)
                skipped  += len([err for err in e.details.get('writeErrors', []) if err.get('code') == 11000])
            batch = []
    except Exception as e:
        errors += 1
        print(f'❌ Error on {json_path.name}: {e}')

# Flush remaining
if batch:
    try:
        result = collection.insert_many(batch, ordered=False)
        inserted += len(result.inserted_ids)
    except BulkWriteError as e:
        inserted += e.details.get('nInserted', 0)
        skipped  += len([err for err in e.details.get('writeErrors', []) if err.get('code') == 11000])

print(f'\n✅ Inserted: {inserted} | Skipped (duplicate): {skipped} | Errors: {errors}')
print(f'📊 Total documents in collection: {collection.count_documents({})}')
print(f'\nSample game_id: {build_game_id(next(iter(csv_index.values())))}')


📄 CSV loaded: 2459 entries
📂 Found 2459 JSON files


Building documents: 100%|██████████| 2459/2459 [01:02<00:00, 39.63it/s] 



✅ Inserted: 2458 | Skipped (duplicate): 1 | Errors: 0
📊 Total documents in collection: 2458

Sample game_id: england_epl/2014-2015/2015-02-21/chelsea-vs-burnley


In [8]:
# ── Cell 5: Inspect / stats ───────────────────────────────────────────────────
total = collection.count_documents({})
print(f'Total documents: {total}')

# Breakdown by league
pipeline = [{'$group': {'_id': '$league', 'count': {'$sum': 1}}}, {'$sort': {'_id': 1}}]
print('\nBy league:')
for r in collection.aggregate(pipeline):
    print(f"  {r['_id']}: {r['count']}")

# Sample document (without raw to keep output clean)
sample = collection.find_one({}, {'raw': 0})
print('\nSample document (no raw field):')
print(json.dumps(sample, indent=2, default=str))

Total documents: 2458

By league:
  england_epl: 647
  europe_champions-league: 569
  france_league-1: 132
  germany_bundesliga: 295
  italy_serie-a: 463
  spain_laliga: 352

Sample document (no raw field):
{
  "_id": "69fb5c2ad63bab50f9f22a8f",
  "game_id": "england_epl/2014-2015/2015-02-21/chelsea-vs-burnley",
  "league": "england_epl",
  "season": "2014-2015",
  "date": "2015-02-21",
  "home_team": "Chelsea",
  "away_team": "Burnley",
  "score": "1 - 1",
  "venue": "Stamford Bridge (London)",
  "referee": "Martin Atkinson"
}
